In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "numpy", "matplotlib", "POT"])


<div style="border-left: 4px solid #1a1a2e; padding: 14px 22px; margin-bottom: 4px; background: #f9f9fc;">
<p style="margin:0 0 4px 0; font-size:0.76em; color:#888; letter-spacing:0.09em; text-transform:uppercase;">Section 4.1</p>
<h1 style="margin:0 0 6px 0; font-size:1.65em; font-weight:700; color:#1a1a2e; line-height:1.2;">Everything is Relative: Understanding Fairness with Optimal Transport</h1>
<p style="margin:0; font-size:0.86em; color:#555;">Kwegyir-Aggrey, K., Santorella, R., Brown, S. M. &nbsp;·&nbsp; arXiv:2102.10349v1</p>
</div>

---

**Scenario.** Blue College admits applicants from two secondary schools under a rule $r \in (0,1)$, the target admission fraction.

| | School A | School B |
|:--|:--|:--|
| **Mechanism** | Each student admitted independently: $P(Y=1\mid X=x) = P(Y=1) = r$ | Top $r$-fraction by GPA admitted: $P(Y=1\mid X=x)=1$ for top $r$, else $0$ |
| **Pattern $\mathcal{F}$** | $\mathcal{F}_1$: point mass at $r$ | $\mathcal{F}_2$: $(1-r)\,\delta_0 + r\,\delta_1$ |

Both schools share $P(Y=1) = r$, so **Disparate Impact** (Def. 4.1, 80% Rule) is not triggered:

$$\frac{P(Y=1 \mid \text{School}=B)}{P(Y=1 \mid \text{School}=A)} = 1 \;\not\leq\; \tau = 0.8 \qquad \Longrightarrow \qquad \text{no disparate impact detected.}$$

Yet the admission *distributions* $\mathcal{F}_1$ and $\mathcal{F}_2$ are structurally different. The **Wasserstein distance** between them remains:

$$W(\mathcal{F}_1, \mathcal{F}_2) = \sqrt{2r(1-r)} \;>\; 0 \qquad \forall\, r \in (0,1)$$

revealing persistent inequity that DI cannot detect.

> **Why does DI oscillate at the start of the simulation?**
>
> The empirical DI is $\widehat{\tau}_t = \min\!\bigl(\hat{p}_{B,t} / \hat{p}_{A,t},\; 1\bigr)$, an asymmetrically clipped ratio.
>
> - *Jumps up from 0:* When $r$ is small and $n$ is small, some years produce $\hat{p}_{A,t} = 0$, forcing $\widehat{\tau}_t = 0$. The running aggregate starts near 0 and climbs.
> - *Drops from 1:* When $r$ is large or $n$ is large, $\hat{p}_{B,t} > \hat{p}_{A,t}$ in many early years (clipped to 1). Once years with $\hat{p}_{B,t} < \hat{p}_{A,t}$ accumulate, the aggregate drifts down to its asymptote.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ot

# ── Style (hidden in HTML export) ─────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 150, "figure.facecolor": "white",
    "axes.facecolor": "white", "axes.edgecolor": "#cccccc", "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#eeeeee", "grid.linewidth": 0.7,
    "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "font.family": "DejaVu Sans", "font.size": 9.5,
    "axes.titlesize": 10.5, "axes.titleweight": "bold", "axes.titlepad": 10,
    "axes.labelsize": 9, "axes.labelcolor": "#333",
    "legend.fontsize": 8.5, "legend.framealpha": 0.9, "legend.edgecolor": "#ddd",
    "lines.linewidth": 1.4,
})
RULE_C = {0.10: "#c0392b", 0.25: "#2471a3", 0.50: "#1e8449"}
N_C    = {50: "#c0392b", 150: "#d68910", 300: "#1e8449", 1000: "#2471a3"}
N_LS   = {50: (4,2), 150: (6,2,1,2), 300: (8,2), 1000: None}

# ── Parameters ────────────────────────────────────────────────────────────
SEED, YEARS = 42, 250
RULES  = [0.10, 0.25, 0.50]
N_LIST = [50, 150, 300, 1000]
GPA_MIN, GPA_MAX = 1.5, 4.0
THRESHOLDS = {r: GPA_MIN + (1 - r) * (GPA_MAX - GPA_MIN) for r in RULES}
THEORY_W   = {r: np.sqrt(2 * r * (1 - r)) for r in RULES}

# ── Simulation ────────────────────────────────────────────────────────────
def simulate_year(rule, n, rng):
    y_A = rng.binomial(1, rule, n).astype(float)
    p_A = y_A.mean()
    y_B = (rng.uniform(GPA_MIN, GPA_MAX, n) >= THRESHOLDS[rule]).astype(float)
    p_B = y_B.mean()
    di  = min(p_B / p_A, 1.0) if p_A > 1e-9 else 0.0
    # W = sqrt(W_1) via sorted-quantile formula — O(n log n), exact for 1-D equal weights
    w   = float(np.sqrt(np.mean(np.abs(np.full(n, rule) - np.sort(y_B)))))
    return di, w

def run_simulation(n):
    rng = np.random.default_rng(SEED)
    res = {r: {"di": np.empty(YEARS), "w": np.empty(YEARS)} for r in RULES}
    for t in range(YEARS):
        for r in RULES:
            res[r]["di"][t], res[r]["w"][t] = simulate_year(r, n, rng)
    return res

cum_mean = lambda a: np.cumsum(a) / np.arange(1, len(a) + 1)

all_results = {n: run_simulation(n) for n in N_LIST}
print("Theory  W =", {r: f"{v:.4f}" for r, v in THEORY_W.items()})


### Figure 2 — Reproduction &nbsp;($n = 150$)

Aggregate DI $\widehat{\tau}$ (left) and per-year $W(\mathcal{F}_1, \mathcal{F}_2)$ (right) over $T = 250$ simulated years, for $r \in \{0.10, 0.25, 0.50\}$.

Under disparate impact, all rules converge to a fair policy. In contrast, $W(\mathcal{F}_1,\mathcal{F}_2)$ recovers the inequities at all rule levels.


In [ ]:
years = np.arange(1, YEARS + 1)
n0    = 150

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
fig.subplots_adjust(wspace=0.38)

# ── Left: Disparate Impact ────────────────────────────────────────────────
ax = axes[0]
ax.axhspan(0.80, 1.00, color="#e8f4f8", zorder=0)
ax.axhline(0.80, color="#2980b9", lw=1.0, ls=(0,(4,3)), alpha=0.8,
           label=r"fair lower-bound  $\tau=0.8$")
ax.axhline(1.00, color="#95a5a6", lw=0.8, ls=(0,(4,3)), alpha=0.7,
           label=r"fair upper-bound  $\tau=1$")
for r in RULES:
    ax.plot(years, cum_mean(all_results[n0][r]["di"]),
            color=RULE_C[r], lw=1.6, label=f"$r = {r}$")
ax.set_xlim(0, YEARS); ax.set_ylim(0.68, 1.04)
ax.set_xlabel("years simulated")
ax.set_ylabel(r"agg. $\hat{\tau}$")
ax.set_title("Disparate Impact")
ax.legend(loc="lower right", fontsize=8)

# ── Right: Wasserstein ────────────────────────────────────────────────────
ax = axes[1]
ax.axhline(0.0, color="#2980b9", lw=0.9, ls=(0,(4,3)), alpha=0.7,
           label="fair lower-bound  $W=0$")
for r in RULES:
    ax.plot(years, all_results[n0][r]["w"],
            color=RULE_C[r], lw=0.9, alpha=0.85, label=f"$r = {r}$")
for r in RULES:
    ax.axhline(THEORY_W[r], color=RULE_C[r], lw=1.0, ls="--", alpha=0.5)
ax.set_xlim(0, YEARS); ax.set_ylim(-0.01, 0.76)
ax.set_xlabel("years simulated")
ax.set_ylabel(r"agg. $W(\mathcal{F}_1,\mathcal{F}_2)$")
ax.set_title("Wasserstein")
ax.legend(loc="upper right", fontsize=7.5)

fig.tight_layout()
plt.show()


### Effect of Sample Size $n$ &nbsp;&nbsp;($r = 0.25$)

The asymptotic values — $\widehat{\tau} \to 1$ and $W(\mathcal{F}_1,\mathcal{F}_2) \to \sqrt{2r(1-r)} = 0.6124$ — are independent of $n$.  
Larger $n$ reduces per-year sampling variance: $\widehat{\tau}$ settles faster and $W_t$ clusters more tightly around the theoretical value.


In [ ]:
r_focus = 0.25
W_TH    = THEORY_W[r_focus]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
fig.subplots_adjust(wspace=0.38)

# ── Left: DI by n ─────────────────────────────────────────────────────────
ax = axes[0]
ax.axhspan(0.80, 1.00, color="#e8f4f8", zorder=0)
ax.axhline(0.80, color="#2980b9", lw=1.0, ls=(0,(4,3)), alpha=0.8,
           label=r"$\tau = 0.8$")
ax.axhline(1.00, color="#95a5a6", lw=0.8, ls=(0,(4,3)), alpha=0.7,
           label=r"$\tau = 1$")
for n in N_LIST:
    kw = {"dashes": N_LS[n]} if isinstance(N_LS[n], tuple) else {"ls": "-"}
    ax.plot(years, cum_mean(all_results[n][r_focus]["di"]),
            color=N_C[n], lw=1.5, label=f"$n = {n}$", **kw)
ax.set_xlim(0, YEARS); ax.set_ylim(0.68, 1.04)
ax.set_xlabel("years simulated")
ax.set_ylabel(r"agg. $\hat{\tau}$")
ax.set_title("Disparate Impact")
ax.legend(fontsize=8)

# ── Right: W by n ─────────────────────────────────────────────────────────
ax = axes[1]
ax.axhline(W_TH, color="#2c3e50", lw=1.8, ls="--", alpha=0.75, zorder=3,
           label=rf"$\sqrt{{2r(1-r)}} = {W_TH:.4f}$")
for n in N_LIST:
    kw = {"dashes": N_LS[n]} if isinstance(N_LS[n], tuple) else {"ls": "-"}
    ax.plot(years, all_results[n][r_focus]["w"],
            color=N_C[n], lw=0.85, alpha=0.8, label=f"$n = {n}$", **kw)
ax.set_xlim(0, YEARS); ax.set_ylim(0.4, 0.8)
ax.set_xlabel("years simulated")
ax.set_ylabel(r"$W(\mathcal{F}_1, \mathcal{F}_2)$")
ax.set_title("Wasserstein Distance")
ax.legend(fontsize=8)

fig.tight_layout()
plt.show()


### Optimal Transport — 4-Panel Visualization &nbsp;($r = 0.25$, $n = 6$)

The four panels show the full OT computation step by step:

- **Panel A — Student GPA**: each of the $n=6$ students' GPA value (sorted descending). Red bars are admitted by School B (GPA ≥ threshold = 3.375), grey bars are rejected. The blue dashed line marks the fixed population threshold.
- **Panel B — $\mathcal{F}_1$ vs $\mathcal{F}_2$ patterns**: side-by-side for each student — $F_1(a_i) = r = 0.25$ (blue, constant for all School A students) vs $F_2(b_j) \in \{0,1\}$ (red, determined by GPA).
- **Panel C — Cost matrix $C$**: $C_{ij} = |F_1(a_i) - F_2(b_j)| = |r - F_2(b_j)|$. Since $F_1$ is constant, every row is identical: cost $= r = 0.25$ for admitted School B students and $= 1-r = 0.75$ for rejected ones.
- **Panel D — Optimal coupling $\pi^*$**: the solution to the OT-LP $\min_{\pi \in \Pi} \sum_{ij} \pi_{ij} C_{ij}$. Non-zero entries show which School A students are "matched" to which School B students; $W_2 = \sqrt{\sum_{ij} \pi^*_{ij} C_{ij}}$.


In [ ]:
VIS_R, VIS_N = 0.25, 6
rng_v = np.random.default_rng(2024)
gpa_v = rng_v.uniform(GPA_MIN, GPA_MAX, VIS_N)

# Sort students by GPA descending for clearer visualization
sort_idx   = np.argsort(gpa_v)[::-1]
gpa_sorted = gpa_v[sort_idx]
y_B_v  = (gpa_sorted >= THRESHOLDS[VIS_R]).astype(float)   # binary: 1=admitted, 0=rejected
F1_v   = np.full(VIS_N, VIS_R)                             # constant r for School A
F2_v   = y_B_v                                              # binary outcomes for School B
C_mat  = np.abs(F1_v[:, None] - F2_v[None, :])             # C_ij = |r - F2_j|
a = b  = np.ones(VIS_N) / VIS_N
pi_mat = ot.emd(a, b, C_mat)
W_v    = float(np.sqrt(np.sum(pi_mat * C_mat)))

gpa_threshold = THRESHOLDS[VIS_R]
n_admitted    = int(y_B_v.sum())
stud_lbl      = [f"s{i+1}" for i in range(VIS_N)]

from matplotlib.patches import Patch

fig = plt.figure(figsize=(14, 10), constrained_layout=True)
fig.suptitle(
    f"OT Visualization  (r={VIS_R}, n={VIS_N})\n"
    f"GPA threshold={gpa_threshold:.2f},  admitted={n_admitted}/{VIS_N},  W\u2082={W_v:.4f}",
    fontsize=12, fontweight="bold")
gs = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.38)

FONT_ANN = 9

# ── Panel A: GPA bar chart ────────────────────────────────────────────────
ax_gpa = fig.add_subplot(gs[0, 0])
bar_colors = ["#e74c3c" if y_B_v[i] == 1 else "#bdc3c7" for i in range(VIS_N)]
bars = ax_gpa.bar(range(VIS_N), gpa_sorted, color=bar_colors,
                  edgecolor="#555", linewidth=0.7, zorder=3)
ax_gpa.axhline(gpa_threshold, color="#2980b9", lw=1.8, ls="--",
               label=f"threshold={gpa_threshold:.2f}")
for i, (bar, g) in enumerate(zip(bars, gpa_sorted)):
    ax_gpa.text(i, g + 0.04, f"{g:.2f}", ha="center", va="bottom",
                fontsize=8.5, fontweight="bold",
                color="#c0392b" if y_B_v[i] == 1 else "#7f8c8d")
ax_gpa.set_xticks(range(VIS_N))
ax_gpa.set_xticklabels(stud_lbl, fontsize=9)
ax_gpa.set_ylim(GPA_MIN - 0.2, GPA_MAX + 0.35)
ax_gpa.set_ylabel("GPA", fontsize=9.5)
ax_gpa.set_title("Panel A  \u2013  Student GPA\n(red = admitted by School B top-r%)", fontsize=9.5)
ax_gpa.yaxis.grid(True, alpha=0.4, zorder=0)
ax_gpa.set_axisbelow(True)
legend_patches = [Patch(facecolor="#e74c3c", label="Admitted (F\u2082=1)"),
                  Patch(facecolor="#bdc3c7", label="Rejected (F\u2082=0)")]
ax_gpa.legend(handles=legend_patches + [plt.Line2D([0],[0],color="#2980b9",ls="--",lw=1.8,
              label=f"threshold={gpa_threshold:.2f}")], fontsize=8, loc="lower right")

# ── Panel B: F1 vs F2 side-by-side bars ───────────────────────────────────
ax_f = fig.add_subplot(gs[0, 1])
x     = np.arange(VIS_N)
width = 0.35
ax_f.bar(x - width/2, F1_v, width, color="#3498db", alpha=0.85,
         edgecolor="#1a5276", linewidth=0.7,
         label=f"F\u2081(a\u1d62) = {VIS_R} (School A, constant)")
ax_f.bar(x + width/2, F2_v, width, color="#e74c3c", alpha=0.85,
         edgecolor="#922b21", linewidth=0.7,
         label="F\u2082(b\u2c7c) \u2208 {0,1} (School B, GPA-based)")
for i in range(VIS_N):
    ax_f.text(i - width/2, F1_v[i] + 0.02, f"{F1_v[i]:.2f}", ha="center",
              va="bottom", fontsize=8.5, color="#1a5276", fontweight="bold")
    ax_f.text(i + width/2, F2_v[i] + 0.02, f"{int(F2_v[i])}", ha="center",
              va="bottom", fontsize=8.5, color="#922b21", fontweight="bold")
ax_f.set_xticks(x)
ax_f.set_xticklabels(stud_lbl, fontsize=9)
ax_f.set_ylim(0, 1.35)
ax_f.set_ylabel("Admission probability / outcome", fontsize=9)
ax_f.set_title("Panel B  \u2013  F\u2081 vs F\u2082 patterns\n"
               "(the two distributions whose W\u2082 we measure)", fontsize=9.5)
ax_f.legend(fontsize=7.5, loc="upper right")
ax_f.yaxis.grid(True, alpha=0.4, zorder=0)
ax_f.set_axisbelow(True)

# helper: annotated matrix grid
def draw_matrix_grid(ax, mat, cmap, vmin, vmax, row_lbl, col_lbl, ann_fmt, title,
                     xlabel="School B student j", ylabel="School A student i"):
    im = ax.imshow(mat, cmap=cmap, aspect="equal",
                   vmin=vmin, vmax=vmax, interpolation="nearest")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04).ax.tick_params(labelsize=7)
    ax.set_xticks(range(len(col_lbl))); ax.set_xticklabels(col_lbl, fontsize=8)
    ax.set_yticks(range(len(row_lbl))); ax.set_yticklabels(row_lbl, fontsize=8)
    ax.set_xlabel(xlabel, fontsize=9); ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(title, fontsize=9.5)
    ax.set_xticks(np.arange(-0.5, len(col_lbl), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(row_lbl), 1), minor=True)
    ax.grid(which="minor", color="#ccc", linestyle="-", linewidth=0.5)
    ax.tick_params(which="minor", bottom=False, left=False)
    for spine in ax.spines.values(): spine.set_visible(False)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            brightness = (v - vmin) / (vmax - vmin + 1e-12)
            ax.text(j, i, ann_fmt(v), ha="center", va="center",
                    fontsize=FONT_ANN, fontweight="bold",
                    color="white" if brightness > 0.55 else "#1a2a3a")

row_lbl   = [f"i={i}  ({stud_lbl[i]})" for i in range(VIS_N)]
col_lbl_m = [f"j={j}  ({stud_lbl[j]})" for j in range(VIS_N)]

# ── Panel C: Cost matrix C ─────────────────────────────────────────────────
ax_c = fig.add_subplot(gs[1, 0])
draw_matrix_grid(
    ax_c, C_mat, cmap="Blues", vmin=0, vmax=1,
    row_lbl=row_lbl, col_lbl=col_lbl_m,
    ann_fmt=lambda v: f"{v:.2f}",
    title=f"Panel C  \u2013  Cost matrix C\n"
          f"C\u1d62\u2c7c = |F\u2081(a\u1d62) \u2212 F\u2082(b\u2c7c)| = |{VIS_R} \u2212 F\u2082|")

# ── Panel D: Optimal transport plan pi* ────────────────────────────────────
ax_pi = fig.add_subplot(gs[1, 1])
draw_matrix_grid(
    ax_pi, pi_mat, cmap="YlOrRd", vmin=0, vmax=pi_mat.max() + 1e-9,
    row_lbl=row_lbl, col_lbl=col_lbl_m,
    ann_fmt=lambda v: f"{v:.3f}" if v > 1e-9 else "0",
    title=r"Panel D  \u2013  Optimal coupling $\pi^*$" + "\n"
          + r"$\sum_{ij}\pi^*_{ij}=1$,  each row/col sums to $1/n$")

plt.show()
